In [50]:
# 1. Импорты, seed, device
import os
import json
import csv
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

import matplotlib.pyplot as plt

In [51]:
# Paths
ROOT = Path("homeworks/HW08-09")
ARTIFACTS = ROOT / "artifacts"
FIGURES = ARTIFACTS / "figures"

ARTIFACTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

In [52]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [53]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cpu')

In [54]:
# 2. Данные и DataLoader (KMNIST)
from torchvision.datasets import EMNIST

BATCH_SIZE = 128

transform = transforms.Compose([
    transforms.ToTensor()
])

dataset_train_full = EMNIST(
    root="data",
    split="balanced",
    train=True,
    download=True,
    transform=transform
)

dataset_test = EMNIST(
    root="data",
    split="balanced",
    train=False,
    download=True,
    transform=transform
)

In [55]:
# Train / Val split (80/20)
val_size = int(0.2 * len(dataset_train_full))
train_size = len(dataset_train_full) - val_size

generator = torch.Generator().manual_seed(SEED)
dataset_train, dataset_val = random_split(
    dataset_train_full,
    [train_size, val_size],
    generator=generator
)

In [56]:
train_loader = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False)

In [57]:
# Sanity check
x, y = next(iter(train_loader))
x.shape, y.shape, x.min().item(), x.max().item()

(torch.Size([128, 1, 28, 28]), torch.Size([128]), 0.0, 1.0)

In [58]:
# 3. MLP модель
class MLP(nn.Module):
    def __init__(self, hidden_sizes, dropout=0.0, batchnorm=False, num_classes=47):
        super().__init__()
        layers = []
        in_features = 28 * 28

        for h in hidden_sizes:
            layers.append(nn.Linear(in_features, h))
            if batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_features = h

        layers.append(nn.Linear(in_features, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.net(x)

In [59]:
# 4. train / eval функции
def accuracy(logits, targets):
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()

In [60]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_acc = 0, 0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += accuracy(logits, y)

    return total_loss / len(loader), total_acc / len(loader)

In [61]:
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_acc = 0, 0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item()
        total_acc += accuracy(logits, y)

    return total_loss / len(loader), total_acc / len(loader)

In [62]:
# 5. EarlyStopping
class EarlyStopping:
    def __init__(self, patience=4):
        self.patience = patience
        self.best = None
        self.counter = 0
        self.stop = False

    def step(self, value):
        if self.best is None or value > self.best:
            self.best = value
            self.counter = 0
            return True
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
            return False

In [63]:
# 6. Универсальный runner эксперимента
def run_experiment(
    experiment_id,
    model_cfg,
    optimizer_cfg,
    max_epochs=20,
    early_stopping=False
):
    model = MLP(**model_cfg, num_classes=47).to(DEVICE)
    criterion = nn.CrossEntropyLoss()

    if optimizer_cfg["name"] == "Adam":
        optimizer = optim.Adam(
            model.parameters(),
            lr=optimizer_cfg["lr"],
            weight_decay=optimizer_cfg.get("weight_decay", 0)
        )
    else:
        optimizer = optim.SGD(
            model.parameters(),
            lr=optimizer_cfg["lr"],
            momentum=optimizer_cfg.get("momentum", 0),
            weight_decay=optimizer_cfg.get("weight_decay", 0)
        )

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    es = EarlyStopping(patience=4) if early_stopping else None

    best_val_acc = 0
    best_state = None

    for epoch in range(max_epochs):
        tl, ta = train_one_epoch(model, train_loader, optimizer, criterion)
        vl, va = evaluate(model, val_loader, criterion)

        history["train_loss"].append(tl)
        history["val_loss"].append(vl)
        history["train_acc"].append(ta)
        history["val_acc"].append(va)

        if va > best_val_acc:
            best_val_acc = va
            best_state = model.state_dict()

        if es:
            improved = es.step(va)
            if improved:
                best_state = model.state_dict()
            if es.stop:
                break

    model.load_state_dict(best_state)
    return model, history, epoch + 1, max(history["val_acc"]), min(history["val_loss"])

In [64]:
# 7. Эксперименты E1–E4
runs = []

In [65]:
# E1 – base
E1_cfg = dict(hidden_sizes=[256, 128], dropout=0.0, batchnorm=False)
model, hist_E1, epochs, best_acc, best_loss = run_experiment(
    "E1", E1_cfg, dict(name="Adam", lr=1e-3)
)
runs.append(("E1", best_acc, best_loss, epochs))

In [66]:
# E2 – Dropout
E2_cfg = dict(hidden_sizes=[256, 128], dropout=0.3, batchnorm=False)
model_E2, hist_E2, epochs_E2, acc_E2, loss_E2 = run_experiment(
    "E2", E2_cfg, dict(name="Adam", lr=1e-3)
)
runs.append(("E2", acc_E2, loss_E2, epochs_E2))

In [67]:
# E3 – BatchNorm
E3_cfg = dict(hidden_sizes=[256, 128], dropout=0.0, batchnorm=True)
model_E3, hist_E3, epochs_E3, acc_E3, loss_E3 = run_experiment(
    "E3", E3_cfg, dict(name="Adam", lr=1e-3)
)
runs.append(("E3", acc_E3, loss_E3, epochs_E3))

In [68]:
# Choose best of E2 / E3
best_cfg = E2_cfg if acc_E2 >= acc_E3 else E3_cfg

In [69]:
# E4 – EarlyStopping
best_model, hist_E4, epochs_E4, acc_E4, loss_E4 = run_experiment(
    "E4", best_cfg, dict(name="Adam", lr=1e-3), early_stopping=True
)
runs.append(("E4", acc_E4, loss_E4, epochs_E4))

In [70]:
# 8. Test оценка + сохранение модели
test_loss, test_acc = evaluate(best_model, test_loader, nn.CrossEntropyLoss())
test_acc

0.8434007532742559

In [71]:
torch.save(best_model.state_dict(), ARTIFACTS / "best_model.pt")

In [72]:
with open(ARTIFACTS / "best_config.json", "w") as f:
    json.dump({
        "dataset": "KMNIST",
        "seed": SEED,
        "model": best_cfg,
        "optimizer": "Adam",
        "lr": 1e-3,
        "test_accuracy": test_acc
    }, f, indent=2)

In [73]:
# 9. O1–O3 (LR и SGD)
# O1 – LR too big
_, hist_O1, _, _, _ = run_experiment(
    "O1", best_cfg, dict(name="Adam", lr=1e-1), max_epochs=6
)

# O2 – LR too small
_, hist_O2, _, _, _ = run_experiment(
    "O2", best_cfg, dict(name="Adam", lr=1e-5), max_epochs=6
)

# O3 – SGD + momentum + weight decay
_, hist_O3, _, acc_O3, _ = run_experiment(
    "O3",
    best_cfg,
    dict(name="SGD", lr=1e-2, momentum=0.9, weight_decay=1e-4),
    max_epochs=12
)

In [74]:
# 10. Графики
plt.figure()
plt.plot(hist_E4["train_loss"], label="train")
plt.plot(hist_E4["val_loss"], label="val")
plt.legend()
plt.title("Best model loss")
plt.savefig(FIGURES / "curves_best.png")
plt.close()

In [75]:
plt.figure(figsize=(8,4))
plt.plot(hist_O1["train_loss"], label="LR too big")
plt.plot(hist_O2["train_loss"], label="LR too small")
plt.legend()
plt.title("LR extremes")
plt.savefig(FIGURES / "curves_lr_extremes.png")
plt.close()

In [76]:
# 11. runs.csv
with open(ARTIFACTS / "runs.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "experiment_id", "dataset", "seed", "best_val_accuracy",
        "best_val_loss", "epochs_trained"
    ])
    for r in runs:
        writer.writerow([r[0], "KMNIST", SEED, r[1], r[2], r[3]])